In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.optimize import minimize

In [2]:
DATA_DIR = Path("output")

OUTPUT_DIR = Path("model_outputs")

RANDOM_STATE = 42

In [3]:
WHEN_CSV_PATTERN = "dataset_winsize*_when.csv"


def convert_when_to_where_dataset_path(when_dataset_path):
    """Return the matching where-dataset path for a when-dataset CSV path."""
    suffix = "_when.csv"

    return when_dataset_path.with_name(
        f"{when_dataset_path.name[:-len(suffix)]}_where.csv"
    )


when_csv_files = sorted(
    DATA_DIR.glob(WHEN_CSV_PATTERN),
    key=lambda p: float(p.stem.split("winsize", 1)[1].removesuffix("h_when")),
)


when_csv_files

[PosixPath('output/dataset_winsize0.25h_when.csv'),
 PosixPath('output/dataset_winsize0.5h_when.csv'),
 PosixPath('output/dataset_winsize1h_when.csv'),
 PosixPath('output/dataset_winsize2h_when.csv'),
 PosixPath('output/dataset_winsize3h_when.csv'),
 PosixPath('output/dataset_winsize4h_when.csv'),
 PosixPath('output/dataset_winsize5h_when.csv'),
 PosixPath('output/dataset_winsize6h_when.csv'),
 PosixPath('output/dataset_winsize7h_when.csv'),
 PosixPath('output/dataset_winsize8h_when.csv'),
 PosixPath('output/dataset_winsize9h_when.csv'),
 PosixPath('output/dataset_winsize10h_when.csv'),
 PosixPath('output/dataset_winsize11h_when.csv'),
 PosixPath('output/dataset_winsize12h_when.csv')]

In [4]:
def load_when_where_datasets(when_dataset_path):
    where_dataset_path = convert_when_to_where_dataset_path(when_dataset_path)

    when_df = pd.read_csv(when_dataset_path)
    where_df = pd.read_csv(where_dataset_path)

    return when_df, where_df


when_df, where_df = load_when_where_datasets(when_csv_files[0])

In [5]:
# check if fold_id columns are equal
all(when_df.fold_id.to_numpy() == where_df.fold_id.to_numpy())

True

In [6]:
# load "t"s
t = when_df["label_time_to_event_seconds"].to_numpy(dtype=float)

T = 24 * 3600
mask = np.isfinite(t) & (t >= 0) & (t <= T)
t_fit = t[mask]

print(f"Number of samples inside {T/3600.0} hours:", len(t_fit))
print(f"Number of samples outside {T/3600.0} hours:", len(t) - len(t_fit))

Number of samples inside 24.0 hours: 5950
Number of samples outside 24.0 hours: 33


In [7]:
def get_truncated_exponential_p(t, T, lambda_):
    """PDF of a truncated exponential distribution."""
    # PDF of an exponential distribution
    num = lambda_ * np.exp(-lambda_ * t)

    # CDF of an exponential distribution
    # denum = 1 - np.exp(-lambda_ * T)
    denum = -np.expm1(-lambda_ * T)

    # Truncated PDF
    truncated_pdf = num / denum

    return truncated_pdf


def get_uniform_p(t, T):
    """PDF of a uniform distribution."""
    return np.ones_like(t) / T


def get_p(t, T, lambda_, p_trunc_exp):
    """Return the probability of a sample t under a mixture of truncated exponential and uniform distributions."""
    p1 = p_trunc_exp * get_truncated_exponential_p(t, T, lambda_)
    p2 = (1 - p_trunc_exp) * get_uniform_p(t, T)

    return p1 + p2


def negative_log_likelihood(params, t, T):
    lambda_, p_trunc_exp = params

    p = get_p(t, T, lambda_, p_trunc_exp)
    p = np.clip(p, 1e-30, None)

    return -np.sum(np.log(p))

In [8]:
x0 = [1 / np.median(t_fit), 0.5]

result = minimize(
    negative_log_likelihood,
    x0=x0,
    args=(t_fit, T),
    bounds=[(1e-8, 1e-2), (0.0, 1.0)],
    method="L-BFGS-B",
)

lambda_hat, p_trunc_exp_hat = result.x

print("Converged:", result.success)

print("p_trunc_exp:", p_trunc_exp_hat)
print("lambda:", lambda_hat)

print("scheduled mean scale, seconds:", 1.0 / lambda_hat)
print("scheduled mean scale, hours:", (1.0 / lambda_hat) / 3600)

Converged: True
p_trunc_exp: 0.8658472033444583
lambda: 0.00017183067271046737
scheduled mean scale, seconds: 5819.682738977505
scheduled mean scale, hours: 1.6165785386048626


In [9]:
weighted_truncated = p_trunc_exp_hat * get_truncated_exponential_p(t, T, lambda_hat)
weighted_background = (1 - p_trunc_exp_hat) * get_uniform_p(t, T)

is_bg = weighted_background > weighted_truncated

min_bg_time = min([j for i, j in zip(is_bg, t) if i]) / 3600
max_fg_time = max([j for i, j in zip(is_bg, t) if not i]) / 3600
threshold = (min_bg_time + max_fg_time) / 2 * 3600

print(min_bg_time, max_fg_time, threshold)

7.4158333333333335 7.333888888888889 26549.5


In [10]:
def add_is_bg_column(df):
    if "is_bg" not in df.columns:
        df = df.copy()
        df["is_bg"] = is_bg

    return df


for when_csv_path in when_csv_files:
    where_csv_path = convert_when_to_where_dataset_path(when_csv_path)
    when_df, where_df = load_when_where_datasets(when_csv_path)

    updated_when_df = add_is_bg_column(when_df)
    updated_where_df = add_is_bg_column(where_df)

    if "is_bg" not in when_df.columns:
        updated_when_df.to_csv(when_csv_path, index=False)

    if "is_bg" not in where_df.columns:
        updated_where_df.to_csv(where_csv_path, index=False)